# PEMS-BAY Traffic Speed Forecasting: GCN + LSTM (spatio-temporal hybrid)

**Idea:** the LSTM models *time* but ignores the road network; the GCN uses the road network but has no real temporal model.
This notebook combines both:

```
speeds of 325 sensors over 12 steps  ->  GCN at every timestep (spatial mixing between neighbouring sensors)
                                      ->  LSTM per sensor over the 12 steps  (temporal modelling)
                                      ->  Linear head  ->  speed of all 325 sensors at the next step (+5 min)
```

**Setup that matches the LSTM notebook** (so the comparison is fair): chronological 70/15/15 split *first*, `StandardScaler` fitted on the **train split only**, 12-step window, next-step prediction, MSE loss, Adam 1e-3, gradient clipping 1.0, best checkpoint chosen on validation loss.

The graph (distance graph from `adj_mx_bay.pkl`, or a correlation graph) is also built from **training data only**, so nothing from the test period leaks in.

## 1. Imports and configuration

In [ ]:
import os, pickle, random, time
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# ----------------------------- configuration -----------------------------
SEQ_LEN        = 12       # 12 steps x 5 min = 60 min of history
BATCH_SIZE     = 64
EPOCHS         = 30
PATIENCE       = 6        # early stopping on validation loss
LR             = 1e-3
GCN_HIDDEN     = 32
LSTM_HIDDEN    = 64
DROPOUT        = 0.1
GRAPH_TYPE     = "distance"   # "distance" = official adj_mx_bay.pkl | "correlation" = Pearson graph (train data only)
CORR_THRESHOLD = 0.5          # only used when GRAPH_TYPE == "correlation"
CONGESTION_MPH = 50.0         # speed < 50 mph  ->  "congested"
SEED           = 42
CHECKPOINT     = "best_gcn_lstm_model.pth"
# -------------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU only (training will be slow)")

## 2. Load data
The notebook looks for the `Data` folder next to it, one level up, or two levels up, so it works from the project root or from a sub-folder like `GCN_LSTM/`.

In [ ]:
def find_file(name):
    candidates = ["Data", "../Data", "../../Data", "/content/drive/MyDrive/data/archive"]
    for base in candidates:
        path = os.path.join(base, name)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Could not find '{name}'. Looked in: {candidates}. Current folder: {os.getcwd()}")

with h5py.File(find_file("pems-bay.h5"), "r") as f:
    speed = f["speed"]["block0_values"][:].astype(np.float32)      # (52116, 325)

with open(find_file("adj_mx_bay.pkl"), "rb") as f:
    _, _, adj_raw = pickle.load(f, encoding="latin1")               # (325, 325) distance-based weights

NUM_NODES = speed.shape[1]
print("Speed matrix:", speed.shape, "| min/max:", speed.min(), speed.max())
print("Adjacency   :", adj_raw.shape)

## 3. Chronological split, scaling, windows

Order matters here: **split -> fit scaler on train -> window each split separately.** This avoids leakage from validation/test into training.

In [ ]:
n         = len(speed)
train_end = int(n * 0.70)
val_end   = train_end + int(n * 0.15)

speed_train, speed_val, speed_test = speed[:train_end], speed[train_end:val_end], speed[val_end:]
print("Raw split  :", speed_train.shape, speed_val.shape, speed_test.shape)

scaler = StandardScaler().fit(speed_train)           # statistics from TRAIN only
train_s, val_s, test_s = (scaler.transform(a).astype(np.float32) for a in (speed_train, speed_val, speed_test))

def make_windows(data, seq_len=SEQ_LEN):
    X = np.stack([data[i:i + seq_len] for i in range(len(data) - seq_len)])   # (N, 12, 325)
    y = data[seq_len:]                                                        # (N, 325)  next step
    return X.astype(np.float32), y.astype(np.float32)

X_train, y_train = make_windows(train_s)
X_val,   y_val   = make_windows(val_s)
X_test,  y_test  = make_windows(test_s)
print("Windows    :", X_train.shape, X_val.shape, X_test.shape)

def loader(X, y, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = loader(X_train, y_train, shuffle=True)
val_loader   = loader(X_val,   y_val,   shuffle=False)
test_loader  = loader(X_test,  y_test,  shuffle=False)
print("Batches    :", len(train_loader), len(val_loader), len(test_loader))

## 4. Build the sensor graph

* `distance` : the official road-distance graph from `adj_mx_bay.pkl` (sparse, physically meaningful).
* `correlation` : Pearson correlation between sensors, computed on the **training split only**. On PEMS-BAY this is usually very dense (hundreds of edges per node), which tends to over-smooth the GCN, so try it as an experiment.

The graph is made symmetric, self-loops are added, and it is normalised as  Â = D^(-1/2) (A + I) D^(-1/2).

In [ ]:
def build_adjacency(kind):
    if kind == "distance":
        A = np.asarray(adj_raw, dtype=np.float32).copy()
    elif kind == "correlation":
        corr = np.nan_to_num(np.corrcoef(speed_train.T))                  # train data only
        A = np.where(corr > CORR_THRESHOLD, corr, 0.0).astype(np.float32)
    else:
        raise ValueError("GRAPH_TYPE must be 'distance' or 'correlation'")
    np.fill_diagonal(A, 0)
    return np.maximum(A, A.T)                                             # make undirected

def normalize_adjacency(A):
    A_hat = A + np.eye(A.shape[0], dtype=np.float32)                      # add self-loops
    d_inv_sqrt = np.power(A_hat.sum(axis=1), -0.5)
    return (d_inv_sqrt[:, None] * A_hat) * d_inv_sqrt[None, :]            # D^-1/2 (A+I) D^-1/2

A = build_adjacency(GRAPH_TYPE)
print(f"Graph: {GRAPH_TYPE} | non-zero edges: {np.count_nonzero(A)} | avg degree: {np.count_nonzero(A) / A.shape[0]:.2f}")

adj_tensor = torch.from_numpy(normalize_adjacency(A)).float().to(device)  # (325, 325)
print("Normalised adjacency:", tuple(adj_tensor.shape))

## 5. Model: GCN -> LSTM

For every timestep the two GCN layers mix each sensor's speed with its neighbours' speeds, `H' = ReLU(Â · H · W)`.
The resulting per-sensor feature sequence (12 steps) is then fed to one LSTM that is **shared across all sensors**, and a linear head turns the last hidden state into the next-step speed.

Shapes: input `(B, 12, 325)` -> `(B, 12, 325, 1)` -> GCN -> `(B, 12, 325, 32)` -> `(B*325, 12, 32)` -> LSTM -> `(B*325, 64)` -> Linear -> `(B, 325)`.

In [ ]:
class GCNLayer(nn.Module):
    """H' = ReLU( A_hat @ (H W) ), applied independently at every timestep."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.lin = nn.Linear(in_features, out_features)

    def forward(self, x, adj):                        # x: (B, T, N, F)   adj: (N, N)
        x = self.lin(x)
        x = torch.einsum("nm,btmf->btnf", adj, x)     # neighbourhood aggregation
        return torch.relu(x)


class GCN_LSTM(nn.Module):
    def __init__(self, gcn_hidden=32, lstm_hidden=64, dropout=0.1):
        super().__init__()
        self.gcn1 = GCNLayer(1, gcn_hidden)
        self.gcn2 = GCNLayer(gcn_hidden, gcn_hidden)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(gcn_hidden, lstm_hidden, batch_first=True)
        self.head = nn.Linear(lstm_hidden, 1)

    def forward(self, x, adj):                        # x: (B, T, N)
        B, T, N = x.shape
        h = x.unsqueeze(-1)                           # (B, T, N, 1)
        h = self.drop(self.gcn1(h, adj))              # (B, T, N, gcn_hidden)
        h = self.gcn2(h, adj)                         # (B, T, N, gcn_hidden)
        h = h.permute(0, 2, 1, 3).reshape(B * N, T, -1)   # one sequence per sensor
        out, _ = self.lstm(h)                         # (B*N, T, lstm_hidden)
        out = self.drop(out[:, -1, :])                # last timestep
        return self.head(out).view(B, N)              # (B, N)


model = GCN_LSTM(GCN_HIDDEN, LSTM_HIDDEN, DROPOUT).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

# quick shape check on one batch
xb, yb = next(iter(train_loader))
with torch.no_grad():
    print("Input:", tuple(xb.shape), "-> Output:", tuple(model(xb.to(device), adj_tensor).shape), "| Target:", tuple(yb.shape))

## 6. Train (early stopping, best checkpoint by validation loss)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def run_epoch(loader, train):
    model.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train:
                optimizer.zero_grad()
            loss = criterion(model(xb, adj_tensor), yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total += loss.item()
    return total / len(loader)

train_losses, val_losses = [], []
best_val, bad_epochs = float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    train_losses.append(tr); val_losses.append(va)

    improved = va < best_val
    if improved:
        best_val, bad_epochs = va, 0
        torch.save(model.state_dict(), CHECKPOINT)
    else:
        bad_epochs += 1

    print(f"Epoch {epoch:2d}/{EPOCHS} | train MSE {tr:.4f} | val MSE {va:.4f} | {time.time() - t0:5.1f}s {'* saved' if improved else ''}")
    if bad_epochs >= PATIENCE:
        print(f"Early stopping: no validation improvement for {PATIENCE} epochs.")
        break

model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
print(f"\nBest validation MSE (scaled): {best_val:.4f}  |  checkpoint: {CHECKPOINT}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses,   label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE (scaled units)")
plt.title("GCN-LSTM training progress")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.5); plt.tight_layout()
plt.savefig("gcn_lstm_loss.png", dpi=150)
plt.show()

## 7. Evaluate on the test set

Predictions are converted back to mph before computing metrics. MAPE ignores true speeds <= 1 mph (division by ~0).
A **"last value" baseline** (predict that the next speed equals the current one) is added because 5-minute-ahead forecasting is easy to get right by persistence: a useful model should beat it.

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    preds = [model(xb.to(device), adj_tensor).cpu().numpy() for xb, _ in loader]
    return np.concatenate(preds)

pred_mph   = scaler.inverse_transform(predict(test_loader))
true_mph   = scaler.inverse_transform(y_test)
last_mph   = scaler.inverse_transform(X_test[:, -1, :])       # persistence baseline

def evaluate(name, pred, true):
    err  = pred - true
    mask = true > 1.0
    yt, yp = (true < CONGESTION_MPH).ravel(), (pred < CONGESTION_MPH).ravel()
    tp = np.sum(yt & yp); fp = np.sum(~yt & yp); fn = np.sum(yt & ~yp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return {
        "Model": name,
        "MAE (mph)":  np.mean(np.abs(err)),
        "RMSE (mph)": np.sqrt(np.mean(err ** 2)),
        "MAPE (%)":   np.mean(np.abs(err[mask]) / true[mask]) * 100,
        "Congestion accuracy (%)": np.mean(yt == yp) * 100,
        "Congestion F1": 2 * prec * rec / max(prec + rec, 1e-12),
    }

results = pd.DataFrame([
    evaluate("Last-value baseline", last_mph, true_mph),
    evaluate("GCN-LSTM (this notebook)", pred_mph, true_mph),
])
results.round(4)

In [ ]:
yt = (true_mph < CONGESTION_MPH).astype(int).ravel()
yp = (pred_mph < CONGESTION_MPH).astype(int).ravel()
print(f"Congestion = speed < {CONGESTION_MPH:.0f} mph\n")
print(classification_report(yt, yp, target_names=["Free flow", "Congested"], digits=4))

## 8. Actual vs predicted (Sensor 0, first 200 test steps)

In [ ]:
sensor = 0
plt.figure(figsize=(12, 5))
plt.plot(true_mph[:200, sensor], label="Actual", linewidth=2)
plt.plot(pred_mph[:200, sensor], label="GCN-LSTM prediction", linestyle="--")
plt.axhline(CONGESTION_MPH, color="gray", linestyle=":", label=f"Congestion boundary ({CONGESTION_MPH:.0f} mph)")
plt.xlabel("Test time step (5 min each)"); plt.ylabel("Speed (mph)")
plt.title(f"GCN-LSTM traffic speed prediction - Sensor {sensor}")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.5); plt.tight_layout()
plt.savefig("gcn_lstm_prediction_sensor0.png", dpi=150)
plt.show()

## 9. Comparison with your earlier models

The LSTM and GCN numbers below are copied from your README files. **Caveat:** your GCN run fitted the scaler and the correlation graph on the *whole* dataset (including the test period), so its numbers are slightly optimistic and not strictly comparable. Re-run the GCN with a train-only scaler/graph for a clean comparison.

In [ ]:
previous = pd.DataFrame([
    {"Model": "LSTM (from LSTM.md)",                        "MAE (mph)": 2.5305, "RMSE (mph)": 4.2249, "MAPE (%)": np.nan, "Congestion F1": np.nan},
    {"Model": "GCN, correlation graph (from GCN.md; leaky)", "MAE (mph)": 2.9027, "RMSE (mph)": 5.1211, "MAPE (%)": 6.69,   "Congestion F1": 0.7515},
])
comparison = pd.concat([previous, results[["Model", "MAE (mph)", "RMSE (mph)", "MAPE (%)", "Congestion F1"]]], ignore_index=True)
comparison.round(4)

## Notes for your report

* **Why this model:** LSTM captures temporal dependence only; GCN captures spatial dependence only; GCN-LSTM captures both, which is the T-GCN-style idea.
* **Possible next steps:** predict 15/30/60 min ahead (3/6/12 steps), add time-of-day features, try `GRAPH_TYPE = "correlation"`, replace the LSTM with a GRU, or add attention.